# Interactive SIR model

Select the `simulation/.venv/bin/python` kernel. After installing with `uv sync --extra notebook`, restart the kernel. Run the two cells below, then control the simulation in the plot.

Computation uses Numba on the CPU. The first step may take longer because of compilation. Restart the kernel after updating the environment.

In [ ]:
%matplotlib widget
from simulation import explore

In [ ]:
explorer = explore(
    grid_shape=(100, 100),
    infection_probability=0.25,
    recover_probability=0.05,
    backend="numba",  
    steps=1000,
    interval=10,
    steps_per_frame=1,
)

Left-click grid cells to add or remove initial infections. Right-click to remove an infection. Disable the pan/zoom tools before selecting cells.

- **Start / Pause** starts and pauses computation.
- **Step** pauses playback and advances exactly one step.
- **Delay [ms]**: higher values slow down the simulation.
- **Step limit**: enter a number and press Enter. You can increase the limit after stopping. This is model time, not seconds.
- **Reset** restores the selected initial infections and enables editing.
- **Clear infections** removes initial infections before starting.

Do not rerun the `explore` cell while the panel is active. To open a new experiment, call `explorer.close()` first.

In [ ]:
# Run while paused: retrieve data without advancing the model
result = explorer.result
print("Completed steps:", explorer.current_step)
print("S, I, R:", result.counts[-1])

## More examples: a recorded run

The cells below demonstrate the earlier API. Close the control panel before continuing.

In [ ]:
explorer.close()
%matplotlib inline

In [ ]:
from simulation import create_simulation, run
import matplotlib.pyplot as plt

In [ ]:
# Parameters to experiment with. Use the same seed to reproduce the experiment.
parameters = dict(
    grid_shape=(40, 40),
    infection_probability=0.25,
    recover_probability=0.05,
    initial_infected=4,
    seed=42,
)
steps = 100
result = run(create_simulation(**parameters), steps=steps)

In [ ]:
result.display(interval=100)

## Selected step and recorded data

`plot` returns a Matplotlib figure and axes, so you can change titles and styling or save the plot. The columns of `counts` are susceptible, infected, and recovered. `grids` contains cell states: 1, 2, 3.

In [ ]:
figure, axes = result.plot(step=25)
# figure.savefig("sir.png", dpi=150)
plt.show()
print("Peak infections:", result.counts[:, 1].max())
print("Peak step:", result.counts[:, 1].argmax())

## Compare infection probabilities

Each experiment creates a new simulation. `run` advances the supplied simulation by the requested number of steps; replaying a recorded animation does not change the model.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for probability in (0.1, 0.25, 0.5):
    config = {**parameters, "infection_probability": probability}
    experiment = run(create_simulation(**config), steps=steps)
    ax.plot(experiment.steps, experiment.counts[:, 1], label=f"p = {probability}")
ax.set(xlabel="Step", ylabel="Infected", title="Effect of infection probability")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## Measure computation without plotting

`benchmark` leaves the supplied model unchanged; every sample starts from the same copy of the grid and random generator. Warmup and compilation are excluded from the samples. Stop the control panel before measuring to reduce CPU contention. `run_timed` changes the model like `run` and also measures history recording. Display the result in a separate cell.

In [ ]:
from simulation import benchmark, run_timed, create_simulation

model = create_simulation((100, 100), initial_infected=100, seed=42)
measurement = benchmark(model, steps=500, repeats=5)
print(f"Median: {measurement.median_seconds:.4f} s")
print(f"Per step: {measurement.milliseconds_per_step:.4f} ms")
print(f"Warmup (including compilation if needed): {measurement.warmup_seconds:.4f} s")

measured = run_timed(model, steps=500)
print(f"Computation: {measured.compute_seconds:.4f} s")
print(f"Recording: {measured.recording_seconds:.4f} s")
print(f"Total excluding warmup and plotting: {measured.total_seconds:.4f} s")